# 02. Financial Deep Dive — 재무 심화 분석

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eddmpython/dartlab/blob/master/notebooks/tutorials/02_financial.ipynb)

정기보고서에 포함된 재무 데이터를 깊이 있게 분석한다. 모든 데이터는 property 한 줄로 접근한다.

**학습 내용:**
- 배당 시계열 (DPS, 배당수익률, 배당성향)
- 직원 현황 (인원, 평균연봉, 근속연수)
- 최대주주와 지분 구조
- 주주 종합 현황
- 주식의 총수
- 부문별 매출
- 비용의 성격별 분류
- 채무증권과 타법인 출자

In [ ]:
!pip install -q dartlab

In [ ]:
from dartlab import Company

samsung = Company("005930")

## 1. 배당

DPS, 배당수익률, 배당성향 등을 연도별로 확인한다. property로 바로 DataFrame을 받는다.

In [ ]:
samsung.dividend

## 2. 직원 현황

직원수, 평균연봉, 근속연수의 시계열이다.

In [ ]:
samsung.employee

## 3. 최대주주

최대주주명, 지분율, 특수관계인 목록을 조회한다. property는 시계열 DataFrame, `get()`으로 상세 정보를 확인한다.

In [ ]:
holder = samsung.get("majorHolder")

print(f"최대주주: {holder.majorHolder} ({holder.majorRatio}%)")
print(f"특수관계인 합계: {holder.totalRatio}%")
print("\n--- 특수관계인 ---")
for h in holder.holders:
    print(f"  {h.name} ({h.relation}): {h.ratioEnd}%")

In [ ]:
samsung.majorHolder

## 4. 주주 종합 현황

5% 이상 주주, 소액주주, 의결권 현황이다. property는 Result 객체를 직접 반환한다.

In [ ]:
overview = samsung.holderOverview

if overview:
    print("--- 5% 이상 주주 ---")
    for bh in overview.bigHolders:
        print(f"  {bh.name}: {bh.ratio}%")

    if overview.minority:
        print(f"\n소액주주 비율: {overview.minority.ratio}%")

    if overview.voting:
        print(f"의결권 행사 가능: {overview.voting.exercisableShares}주")

## 5. 주식의 총수

발행주식, 자기주식, 유통주식의 시계열. `get()`으로 상세 필드도 접근 가능하다.

In [ ]:
cap = samsung.get("shareCapital")

if cap:
    print(f"발행한 주식의 총수: {cap.issuedShares:,.0f}")
    print(f"자기주식: {cap.treasuryShares:,.0f}")
    print(f"유통주식: {cap.outstandingShares:,.0f}")
    print(f"자사주 비율: {cap.treasuryRatio:.2%}")

In [ ]:
samsung.shareCapital

## 6. 부문별 매출

K-IFRS 주석의 부문정보를 Notes로 접근한다.

In [ ]:
samsung.notes.segments

## 7. 비용의 성격별 분류

원재료비, 인건비, 감가상각비 등 비용 항목별 시계열과 구성비.

In [ ]:
cost = samsung.get("costByNature")

if cost:
    print("--- 비용 시계열 ---")
    print(cost.timeSeries)
    print("\n--- 구성비 ---")
    print(cost.ratios)

## 8. 채무증권 발행실적

`get()`으로 개별 발행 내역도 확인 가능하다.

In [ ]:
b = samsung.get("bond")

if b:
    for issuance in b.issuances[:5]:
        print(f"{issuance.bondType} | {issuance.amount}백만원 | {issuance.interestRate}")
    print("\n--- 시계열 ---")
    print(b.timeSeries)

## 9. 타법인 출자

자회사 투자 종합 현황. `get()`으로 개별 투자법인 접근 가능.

In [ ]:
sub = samsung.get("subsidiary")

if sub:
    for inv in sub.investments[:5]:
        print(f"{inv.name}: {inv.endRatio}%, 장부가 {inv.endBook}")
    print("\n--- 시계열 ---")
    print(sub.timeSeries)